# Running Tau-Bench (Agentic Benchmark) with Inspect AI on SageMaker

## What is Tau-Bench?

[Tau-Bench](https://github.com/sierra-research/tau2-bench) (Tau2) is an agentic benchmark that evaluates language models on realistic, multi-turn tool-use tasks. It simulates customer service scenarios where the model must:

- Interact with a simulated user over multiple conversation turns
- Use tools (APIs) to look up information and perform actions
- Follow domain-specific policies and procedures
- Complete tasks accurately without violating constraints

### Domains

Tau2 includes **3 evaluation domains**:

| Domain | Description | Scoring |
|--------|-------------|---------|
| `tau2_airline` | Airline customer service (booking changes, cancellations, policy questions) | Tool calls + natural language goal assessment via grading model |
| `tau2_retail` | Retail customer service (order tracking, returns, product inquiries) | Tool calls + information relay verification |
| `tau2_telecom` | Telecom customer service (plan changes, billing, troubleshooting) | Tool calls + end environment state comparison |

### ⚠️ Token Usage Warning

This benchmark uses a **high amount of tokens**. For example, the airline domain on a single model may use ~60M tokens. For testing purposes, use:
- `--limit 5` to limit the number of samples
- `-T message_limit=10` to limit the number of messages per sample

## What is Inspect AI?

[Inspect AI](https://inspect.ai-safety-institute.org.uk/) is an open-source framework for LLM evaluations created by the UK AI Safety Institute. It provides:

- Standardized evaluation tasks (multiple choice, generation, code execution, **agent-based**)
- Diverse scoring methods and parallel execution
- Rich logging and visualization of results
- Extensible model provider system (OpenAI, Anthropic, **SageMaker**, etc.)

## Prerequisites

- **AWS account** with a SageMaker endpoint deployed (running vLLM or OpenAI-compatible inference)
- **AWS credentials** configured (via AWS CLI, environment variables, or IAM role)
- **Python 3.12 or higher**
- **SageMaker endpoint** that supports the OpenAI Chat Completions API format with tool calling

## Step 1: Set Up Environment and Install Dependencies

Create a Python virtual environment and install the required packages. This ensures the `inspect` CLI and all dependencies (including `boto3`) are available.

### Create and activate a virtual environment

If you haven't already set up a virtual environment for this notebook:

In [ ]:
# Create venv (run once)
!python3.12 -m venv .venv

# Install ipykernel to use this venv as a Jupyter kernel
!.venv/bin/pip install ipykernel
!.venv/bin/python -m ipykernel install --user --name=inspect-eval --display-name="Python (inspect-eval)"

# NOTE: After running this cell, select the "Python (inspect-eval)" kernel
# from the kernel picker in your notebook interface.

### Install packages

In [ ]:
!pip install uv

In [ ]:
# Install core evaluation framework and benchmarks
!uv pip install inspect-ai inspect-evals

In [ ]:
# Install AWS dependencies for SageMaker provider
!uv pip install aioboto3 boto3 botocore openai

## Step 2: Configure AWS Credentials

Ensure your AWS credentials are properly configured. The SageMaker provider uses boto3 to authenticate.

### Configuration Options

1. **AWS CLI Configuration** (Recommended for local development)
   - Run `aws configure` and provide your credentials

2. **Environment Variables**
   ```bash
   export AWS_ACCESS_KEY_ID=your_access_key
   export AWS_SECRET_ACCESS_KEY=your_secret_key
   export AWS_DEFAULT_REGION=us-west-2
   ```

3. **IAM Role** (For EC2/SageMaker Notebooks)
   - Automatically uses the instance's IAM role

### Required IAM Permissions

Your IAM user/role needs `sagemaker:InvokeEndpoint` permission on your endpoint.

### Verify Configuration

In [ ]:
import boto3

# Verify AWS credentials
try:
    sts = boto3.client('sts')
    identity = sts.get_caller_identity()
    print(f"✓ AWS credentials configured")
    print(f"  Account: {identity['Account']}")
    print(f"  User/Role: {identity['Arn']}")
except Exception as e:
    print(f"✗ AWS credentials not configured: {e}")
    print("\nPlease configure AWS credentials using one of:")
    print("  - AWS CLI: aws configure")
    print("  - Environment variables: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY")
    print("  - IAM role (if running on AWS)")

## Step 3: Install the SageMaker Provider

### What is the SageMaker Provider?

The SageMaker provider is a custom Inspect AI model provider that enables communication between Inspect AI and your SageMaker endpoints. It:

1. **Translates Inspect AI requests** into the OpenAI Chat Completions API format expected by your endpoint
2. **Handles AWS authentication** using boto3/aioboto3 to securely invoke endpoints
3. **Manages retries and error handling** for robust evaluation runs
4. **Supports tool calling** required for agentic benchmarks like tau-bench

### How It Works

When you run an evaluation:
- Inspect AI calls the provider with evaluation samples
- The provider formats requests and invokes your SageMaker endpoint via `sagemaker-runtime.invoke_endpoint()`
- Responses are parsed and returned to Inspect AI for scoring

### Installation

The cells below will:
1. Locate your Inspect AI installation
2. Create the `sagemaker.py` provider file
3. Register it so you can use `--model sagemaker/your-endpoint-name`

In [ ]:
import os
import sys
from pathlib import Path

# Find the Inspect AI installation directory
try:
    import inspect_ai

    if hasattr(inspect_ai, '__file__') and inspect_ai.__file__:
        inspect_ai_path = os.path.dirname(inspect_ai.__file__)
    else:
        inspect_ai_path = str(Path(inspect_ai.__path__[0]))

    providers_dir = os.path.join(inspect_ai_path, 'model', '_providers')
    os.makedirs(providers_dir, exist_ok=True)

    print(f"✓ Found Inspect AI at: {inspect_ai_path}")
    print(f"  Providers directory: {providers_dir}")

except ImportError:
    print("✗ Inspect AI not found. Please install it first (Step 1)")
    sys.exit(1)

### Define the SageMaker Provider Code

The following cell contains the full provider implementation. It handles:
- Message formatting (system, user, assistant, tool messages)
- Tool call serialization/deserialization
- vLLM-specific configuration
- Retry logic for transient SageMaker errors (500, 503, 504)

In [ ]:
sagemaker_provider_code = '''"""AWS SageMaker model provider for Inspect AI."""

import json
from logging import getLogger
from typing import Any
from botocore.config import Config
from botocore.exceptions import ClientError
from openai.types.chat import ChatCompletion
from typing_extensions import override
from inspect_ai._util.constants import DEFAULT_MAX_TOKENS
from inspect_ai._util.content import Content
from inspect_ai._util.error import pip_dependency_error
from inspect_ai._util.images import file_as_data_uri
from inspect_ai._util.url import is_http_url
from inspect_ai._util.version import verify_required_version
from inspect_ai.model._openai import chat_choices_from_openai, model_output_from_openai
from inspect_ai.tool import ToolChoice, ToolInfo
from inspect_ai.tool._tool_choice import ToolFunction
from inspect_ai.model._chat_message import ChatMessage, ChatMessageAssistant, ChatMessageSystem, ChatMessageTool, ChatMessageUser
from inspect_ai.model._generate_config import GenerateConfig
from inspect_ai.model._model import ModelAPI
from inspect_ai.model._model_call import ModelCall
from inspect_ai.model._model_output import ModelOutput

logger = getLogger(__name__)

SAGEMAKER_DEFAULTS = {"region_name": "us-east-1", "read_timeout": 600, "connect_timeout": 60}
SAGEMAKER_RETRY_ERROR_CODES = {0, 500, 503, 504}

class SagemakerAPI(ModelAPI):
    def __init__(self, model_name: str, config: GenerateConfig = GenerateConfig(), **model_args: Any):
        super().__init__(model_name=model_name, base_url=None, api_key=None, api_key_vars=[], config=config)
        self.endpoint_name = model_name
        self.model_args = SAGEMAKER_DEFAULTS | model_args
        try:
            import aioboto3
            verify_required_version("Sagemaker API", "aioboto3", "13.0.0")
            self.session = aioboto3.Session()
        except ImportError:
            raise pip_dependency_error("Sagemaker API", ["aioboto3"])
        self.request_content_type = "application/json"
        self.request_accept_type = "application/json"

    @override
    def connection_key(self) -> str:
        return self.endpoint_name

    @override
    def max_tokens(self) -> int | None:
        return DEFAULT_MAX_TOKENS

    @override
    def should_retry(self, ex: Exception) -> bool:
        if isinstance(ex, ClientError):
            error_code = ex.response.get("Error", {}).get("Code", "")
            status_code = ex.response.get("OriginalStatusCode", -1)
            return error_code == "ModelError" and status_code in SAGEMAKER_RETRY_ERROR_CODES
        return False

    @override
    def collapse_user_messages(self) -> bool:
        return True

    @override
    def collapse_assistant_messages(self) -> bool:
        return True

    async def generate(self, input: list[ChatMessage], tools: list[ToolInfo], tool_choice: ToolChoice, config: GenerateConfig):
        config = self._prepare_vllm_config(input, config)
        tools_config = self._prepare_tools_config(tools)
        processed_messages = await self._prepare_messages(input)
        request_body = self._build_request_body(config, processed_messages, tools_config, tool_choice)
        async with self._create_client() as client:
            body_bytes = await self._invoke_endpoint(client, request_body)
        output = json.loads(body_bytes.decode("utf-8"))
        model_output = model_output_from_response(output, tools)
        model_call = ModelCall.create(request=request_body, response=output, time=0)
        return model_output, model_call

    def _prepare_vllm_config(self, input: list[ChatMessage], config: GenerateConfig) -> GenerateConfig:
        if not (input and isinstance(input[-1], ChatMessageAssistant)):
            return config
        config = config.model_copy()
        if config.extra_body is None:
            config.extra_body = {}
        config.extra_body.setdefault("add_generation_prompt", False)
        config.extra_body.setdefault("continue_final_message", True)
        return config

    def _prepare_tools_config(self, tools: list[ToolInfo]):
        if not tools:
            return None
        return [{"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.parameters.model_dump(exclude_none=True)}} for t in tools]

    async def _prepare_messages(self, input: list[ChatMessage]):
        collapsed = collapse_consecutive_messages(input, self.collapse_user_messages(), self.collapse_assistant_messages())
        return [await process_chat_message(message) for message in collapsed]

    def _create_client(self):
        return self.session.client(service_name="sagemaker-runtime", region_name=self.model_args["region_name"], endpoint_url=self.model_args.get("endpoint_url"), config=Config(read_timeout=self.model_args["read_timeout"], connect_timeout=self.model_args["connect_timeout"], retries={"total_max_attempts": 1, "mode": "standard"}))

    def _build_request_body(self, config: GenerateConfig, messages, tools_config, tool_choice: ToolChoice):
        request_body = {"messages": messages, "max_tokens": config.max_tokens, "temperature": config.temperature, "top_p": config.top_p}
        self._add_optional_params(request_body, config)
        if tools_config:
            request_body["tools"] = tools_config
            self._add_tool_choice(request_body, tool_choice)
            if config.parallel_tool_calls is not None:
                request_body["parallel_tool_calls"] = config.parallel_tool_calls
        if config.response_schema is not None:
            request_body["response_format"] = {"type": "json_schema", "json_schema": {"name": config.response_schema.name, "schema": config.response_schema.json_schema.model_dump(exclude_none=True), "description": config.response_schema.description, "strict": config.response_schema.strict}}
        if config.extra_body:
            request_body.update(config.extra_body)
        return request_body

    def _add_optional_params(self, request_body, config: GenerateConfig):
        for k, v in [("top_k", config.top_k), ("stop", config.stop_seqs), ("frequency_penalty", config.frequency_penalty), ("presence_penalty", config.presence_penalty), ("logit_bias", config.logit_bias), ("seed", config.seed), ("n", config.num_choices), ("logprobs", config.logprobs), ("top_logprobs", config.top_logprobs), ("best_of", config.best_of), ("reasoning_effort", config.reasoning_effort)]:
            if v is not None:
                request_body[k] = v

    def _add_tool_choice(self, request_body, tool_choice: ToolChoice):
        if isinstance(tool_choice, ToolFunction):
            request_body["tool_choice"] = {"type": "function", "function": {"name": tool_choice.name}}
        elif tool_choice == "any":
            request_body["tool_choice"] = "required"
        elif tool_choice == "none":
            request_body["tool_choice"] = "none"
        else:
            request_body["tool_choice"] = "auto"

    async def _invoke_endpoint(self, client, request_body):
        response = await client.invoke_endpoint(EndpointName=self.endpoint_name, ContentType=self.request_content_type, Accept=self.request_accept_type, Body=json.dumps(request_body))
        return await response["Body"].read()

async def process_chat_message(message: ChatMessage):
    if isinstance(message, (ChatMessageSystem, ChatMessageUser)):
        content = await process_content(message.content)
        return {"role": message.role, "content": content}
    elif isinstance(message, ChatMessageAssistant):
        content = await process_content(message.content)
        result = {"role": message.role, "content": content}
        if message.tool_calls:
            result["tool_calls"] = [{"id": tc.id, "type": "function", "function": {"name": tc.function, "arguments": json.dumps(tc.arguments)}} for tc in message.tool_calls]
        return result
    elif isinstance(message, ChatMessageTool):
        content = f"Error: {message.error.message}" if message.error else message.text
        return {"role": "tool", "tool_call_id": str(message.tool_call_id), "content": content}
    else:
        raise ValueError(f"Unexpected message type: {type(message)}")

async def process_content(content):
    if isinstance(content, str):
        return content
    processed = []
    for item in content:
        if item.type == "text":
            processed.append({"type": "text", "text": item.text})
        elif item.type == "image":
            image_url = item.image if is_http_url(item.image) else await file_as_data_uri(item.image)
            processed.append({"type": "image_url", "image_url": {"url": image_url, "detail": getattr(item, "detail", "auto")}})
        elif item.type == "reasoning":
            processed.append({"type": "reasoning", "reasoning": item.reasoning})
    if len(processed) == 1 and processed[0]["type"] == "text":
        return processed[0]["text"]
    return processed

def collapse_consecutive_messages(messages, collapse_user, collapse_assistant):
    if not messages:
        return []
    collapsed = [messages[0]]
    for msg in messages[1:]:
        last = collapsed[-1]
        if msg.role == last.role and ((isinstance(msg, ChatMessageUser) and collapse_user) or (isinstance(msg, ChatMessageAssistant) and collapse_assistant)):
            last.content.extend(msg.content)
        else:
            collapsed.append(msg)
    return collapsed

def model_output_from_response(output, tools: list[ToolInfo]):
    completion = ChatCompletion.model_validate(output)
    choices = chat_choices_from_openai(completion, tools)
    return model_output_from_openai(completion, choices)
'''

### Write and Register the Provider

In [ ]:
# Write the provider file and register it with Inspect AI

if sagemaker_provider_code:
    # Write the provider file
    sagemaker_file = os.path.join(providers_dir, 'sagemaker.py')
    with open(sagemaker_file, 'w') as f:
        f.write(sagemaker_provider_code)
    print(f"✓ SageMaker provider installed at: {sagemaker_file}")

    # Register the provider in providers.py
    providers_file = os.path.join(providers_dir, 'providers.py')
    with open(providers_file, 'r') as f:
        providers_content = f.read()

    if '@modelapi(name="sagemaker")' not in providers_content:
        bedrock_end = providers_content.find('@modelapi(name="mockllm")')
        if bedrock_end > 0:
            sagemaker_registration = '''\n\n@modelapi(name="sagemaker")
def sagemaker() -> type[ModelAPI]:
    from .sagemaker import SagemakerAPI

    return SagemakerAPI


'''
            new_content = providers_content[:bedrock_end] + sagemaker_registration + providers_content[bedrock_end:]
            with open(providers_file, 'w') as f:
                f.write(new_content)
            print(f"✓ SageMaker provider registered in: {providers_file}")
        else:
            print("⚠ Could not find insertion point in providers.py")
    else:
        print("✓ SageMaker provider already registered")

    print("\n✓ Installation complete! You can now use model='sagemaker/your-endpoint-name'")
else:
    print("✗ No provider code available.")

## Step 4: Run Tau-Bench Evaluation

### Available Tasks

| Task | Command | Description |
|------|---------|-------------|
| Airline | `inspect_evals/tau2_airline` | Booking changes, cancellations, policy questions |
| Retail | `inspect_evals/tau2_retail` | Order tracking, returns, product inquiries |
| Telecom | `inspect_evals/tau2_telecom` | Plan changes, billing, troubleshooting |

### Parameters

| Parameter | Description | Recommended |
|-----------|-------------|-------------|
| `--model` | SageMaker endpoint: `sagemaker/endpoint-name` | Your endpoint |
| `-M region_name` | AWS region of your endpoint | Match your deployment |
| `--max-connections` | Parallel requests to endpoint | 4-16 for single instance |
| `--max-retries` | Retry attempts for failed requests | 10-50 |
| `--limit` | Number of samples to evaluate | 5 for testing, omit for full |
| `-T message_limit` | Max messages per sample (controls token usage) | 10 for testing |
| `--display plain` | Plain text output (best for notebooks) | Always in notebooks |

### Run the Evaluation

Update the endpoint name and region below to match your deployment:

In [ ]:
# Run tau-bench airline evaluation (quick test with limited samples and messages)
# Update YOUR_ENDPOINT_NAME and region_name to match your SageMaker deployment

!inspect eval inspect_evals/tau2_airline \
    --model sagemaker/YOUR_ENDPOINT_NAME \
    -M region_name=YOUR_REGION \
    --max-connections 4 \
    --max-retries 10 \
    --limit 5 \
    -T message_limit=10 \
    --display plain

To run the full benchmark (⚠️ high token usage), remove `--limit` and `-T message_limit`:

```bash
inspect eval inspect_evals/tau2_airline \\
    --model sagemaker/YOUR_ENDPOINT_NAME \\
    -M region_name=YOUR_REGION \\
    --max-connections 16 \\
    --max-retries 50 \\
    --display plain
```

To run all three domains together:

```bash
inspect eval-set inspect_evals/tau2_airline inspect_evals/tau2_retail inspect_evals/tau2_telecom \\
    --model sagemaker/YOUR_ENDPOINT_NAME \\
    -M region_name=YOUR_REGION \\
    --max-connections 16 \\
    --max-retries 50 \\
    --display plain
```

### View Results

After the evaluation completes, use the Inspect AI viewer to explore detailed results:
- Per-sample conversation traces
- Tool call success/failure rates
- Overall accuracy scores

In [ ]:
# Launch the Inspect AI results viewer
!inspect view

### Interpreting Results

The Inspect AI viewer (`inspect view`) provides a detailed breakdown of your evaluation:

**Overview Panel (Top)**
- **accuracy** — fraction of samples where the agent completed the task correctly
- **stderr** — standard error (high with small sample sizes; use more samples for reliable estimates)
- **tokens** — total token consumption for the run

**Sample List (Left Panel)**

Each row is one tau-bench task, marked as:
- ✓ (green) — task passed (all tool calls correct + goals met)
- ✗ (red) — task failed (wrong tool calls or unmet goals)
- E (error) — task errored out (e.g., endpoint returned an error)

**Sample Detail (Right Panel)**

Click any sample to inspect:
- **Messages** — the full multi-turn conversation (system prompt, user messages, assistant responses, tool calls/results)
- **Scoring** — why the sample passed or failed (expected vs. actual tool calls, goal assessment)
- **Error** — the exception details if the sample errored

**What to Look For**

| Observation | Meaning |
|-------------|---------|
| Many errors (E) | Infrastructure issue — check endpoint compatibility and provider config |
| Failures with wrong tool calls | Model needs better tool-calling fine-tuning |
| Failures with correct tools but unmet goals | Model understands tools but struggles with policy/reasoning |
| High accuracy on retail, low on airline | Airline tasks are harder (require grading model assessment) |

### Reference Results (GPT-5, pass^1 rate)

For comparison, here are the reference scores from the Tau2 leaderboard (Agent: GPT-5, User simulator: gpt-4.1):

| Domain | Inspect Evals Accuracy | Stderr | Tau2 Leaderboard |
|--------|----------------------|--------|------------------|
| Retail | 0.825 | 0.036 | 0.816 |
| Airline | 0.580 | 0.071 | 0.625 |
| Telecom | 0.939 | 0.023 | 0.958 |

*Note: Leaderboard uses 4 epochs (`--epochs 4`) and `message_limit=100`. Inspect Evals accuracy above is 1 epoch with no message limit.*

## Next Steps

1. **Run the full benchmark**: Remove `--limit` and `-T message_limit` flags
2. **Try all domains**: Run airline, retail, and telecom with `eval-set`
3. **Scale up**: Increase `--max-connections` for multi-instance endpoints
4. **Compare models**: Run the same benchmark against different endpoints/checkpoints
5. **Multiple epochs**: Use `--epochs 4` to match the official leaderboard methodology

### Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `Unsupported fields: ['add_generation_prompt']` | Endpoint doesn't support vLLM prompt options | Update provider's `_prepare_vllm_config` to skip these fields |
| `continue_final_message is not supported` | Model doesn't support message continuation | Same as above |
| `list index out of range` | Model returned empty/malformed tool calls | Check model's tool-calling capability |
| `ModelError` with 400 | Request format mismatch | Verify endpoint supports OpenAI Chat Completions format |

### Resources

- [Inspect AI Documentation](https://inspect.ai-safety-institute.org.uk/)
- [Inspect Evals Repository](https://github.com/UKGovernmentBEIS/inspect_evals)
- [Tau2 Paper](https://arxiv.org/abs/2506.07982)
- [Tau-Bench Paper](https://arxiv.org/abs/2406.12045)
- [Tau2 Leaderboard](https://taubench.com/#leaderboard)
- [AWS SageMaker Documentation](https://docs.aws.amazon.com/sagemaker/)